# Bad Fanfic: train on your own books

Runs the neural routes from [christoffellis/bad-fanfic](https://github.com/christoffellis/bad-fanfic) on a free Colab GPU.

Use **Runtime > Change runtime type > T4 GPU** before you start.

Only upload DRM-free epubs you own. Don't publish models trained on copyrighted books: they can reproduce passages from them.

In [ ]:
!git clone --depth 1 https://github.com/christoffellis/bad-fanfic
%cd bad-fanfic
!pip install -q torch transformers "optimum[onnxruntime]"

## 1. Upload your epubs

In [ ]:
import os
from google.colab import files

os.makedirs("books", exist_ok=True)
for name, data in files.upload().items():
    with open(os.path.join("books", name), "wb") as f:
        f.write(data)
print(os.listdir("books"))

## 2. Extract the text

In [ ]:
!python -m bad_fanfic extract books -o corpus.txt

## 3a. Fine-tune GPT-2

A few hundred steps is plenty. Training much longer makes it recite the books instead of mangling them.

In [ ]:
!python -m bad_fanfic gpt2 train corpus.txt --steps 400 --save-every 100

In [ ]:
!python -m bad_fanfic gpt2 generate --model models/gpt2/final --prompt "Chapter One" --temperature 1.0 --count 3

Try an earlier checkpoint for more chaos, for example `--model models/gpt2/step-100`.

## 3b. Or train the character-level LSTM (the 2015 look)

In [ ]:
!python -m bad_fanfic lstm train corpus.txt --steps 3000

In [ ]:
!python -m bad_fanfic lstm generate --prompt "The " --length 600 --temperature 0.8 --count 2

## 4. Optional: export for the browser demo

Only do this for a model trained on text you can legally redistribute (public-domain books, your own writing).

In [ ]:
!python -m bad_fanfic export models/gpt2/final -o export/my-bad-fanfic

In [ ]:
# from huggingface_hub import notebook_login, upload_folder
# notebook_login()
# upload_folder(folder_path="export/my-bad-fanfic", repo_id="YOUR_USER/my-bad-fanfic", repo_type="model", create_repo=True)

Then open the demo with `?model=YOUR_USER/my-bad-fanfic` on the end of the page URL.